# Notebook 4: Integrated Rainfall–Vegetation Analysis in Ghana (2000–2024)

## Objective
Integrate CHIRPS annual rainfall estimates and MODIS annual mean NDVI to investigate relationships between rainfall variability and vegetation greenness in Ghana.

## Research questions
1. How strongly are annual rainfall estimates associated with annual mean NDVI?
2. Is the observed relationship robust to the influence of individual years and long-term trends?
3. What limitations affect interpretation of the relationship as an indicator of potential forage conditions?

## Data sources
- CHIRPS-derived annual rainfall estimates for Ghana, 2000–2024.
- MODIS MOD13Q1-derived annual mean NDVI, 2000–2024.

## Methodological caution
The MODIS spatial-coverage calculation has not yet been independently verified. The annual NDVI results and their relationship with rainfall must therefore be interpreted cautiously. NDVI indicates vegetation greenness, not direct forage biomass or livestock feed availability.

In [2]:

# Description: Upload the annual rainfall and NDVI CSV datasets.

from google.colab import files
import pandas as pd
from io import BytesIO

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print("-", filename)

rainfall_filename = "ghana_annual_rainfall_estimates_2000_2024.csv"
ndvi_filename = "ghana_annual_ndvi_2000_2024.csv"

if rainfall_filename not in uploaded or ndvi_filename not in uploaded:
    print("\nPlease upload both required files:")
    print("-", rainfall_filename)
    print("-", ndvi_filename)
else:
    rainfall_df = pd.read_csv(BytesIO(uploaded[rainfall_filename]))
    ndvi_df = pd.read_csv(BytesIO(uploaded[ndvi_filename]))

    print("\nRainfall shape:", rainfall_df.shape)
    print("NDVI shape:", ndvi_df.shape)

    print("\nRainfall columns:", rainfall_df.columns.tolist())
    print("NDVI columns:", ndvi_df.columns.tolist())

    print("\nRainfall preview:")
    display(rainfall_df.head())

    print("\nNDVI preview:")
    display(ndvi_df.head())


Saving ghana_annual_rainfall_estimates_2000_2024.csv to ghana_annual_rainfall_estimates_2000_2024 (1).csv
Saving ghana_rainfall_ndvi_merged_2000_2024.csv to ghana_rainfall_ndvi_merged_2000_2024.csv
Uploaded files:
- ghana_annual_rainfall_estimates_2000_2024 (1).csv
- ghana_rainfall_ndvi_merged_2000_2024.csv

Please upload both required files:
- ghana_annual_rainfall_estimates_2000_2024.csv
- ghana_annual_ndvi_2000_2024.csv


In [3]:
# Load and inspect the existing rainfall–NDVI dataset
import pandas as pd

rainfall_ndvi = pd.read_csv(
    "/content/ghana_rainfall_ndvi_merged_2000_2024.csv"
)

print("Dataset shape:", rainfall_ndvi.shape)
print("Columns:", rainfall_ndvi.columns.tolist())
display(rainfall_ndvi.head())

Dataset shape: (25, 3)
Columns: ['year', 'annual_rainfall_mm', 'annual_mean_ndvi']


,year,annual_rainfall_mm,annual_mean_ndvi
0,2000,1126.608681,0.553957
1,2001,1043.306209,0.513885
2,2002,1231.959347,0.573142
3,2003,1271.682950,0.566151
4,2004,1164.307682,0.585937


In [4]:
# Test rainfall–NDVI association after removing linear time trends
import numpy as np
from scipy.stats import linregress, pearsonr, spearmanr

# Fit a linear trend to each variable
rain_trend = linregress(rainfall_ndvi["year"], rainfall_ndvi["annual_rainfall_mm"])
ndvi_trend = linregress(rainfall_ndvi["year"], rainfall_ndvi["annual_mean_ndvi"])

# Calculate detrended residuals
rainfall_ndvi["rainfall_residual"] = (
    rainfall_ndvi["annual_rainfall_mm"] -
    (rain_trend.intercept + rain_trend.slope * rainfall_ndvi["year"])
)

rainfall_ndvi["ndvi_residual"] = (
    rainfall_ndvi["annual_mean_ndvi"] -
    (ndvi_trend.intercept + ndvi_trend.slope * rainfall_ndvi["year"])
)

# Correlate the residuals
pearson_r, pearson_p = pearsonr(
    rainfall_ndvi["rainfall_residual"],
    rainfall_ndvi["ndvi_residual"]
)

spearman_rho, spearman_p = spearmanr(
    rainfall_ndvi["rainfall_residual"],
    rainfall_ndvi["ndvi_residual"]
)

print("DE-TRENDED RAINFALL–NDVI CORRELATION")
print("------------------------------------")
print(f"Pearson correlation: r = {pearson_r:.4f}, p = {pearson_p:.4f}")
print(f"Spearman correlation: rho = {spearman_rho:.4f}, p = {spearman_p:.4f}")
print(f"Rainfall trend: {rain_trend.slope:.3f} mm/year")
print(f"NDVI trend: {ndvi_trend.slope:.6f} NDVI units/year")

DE-TRENDED RAINFALL–NDVI CORRELATION
------------------------------------
Pearson correlation: r = 0.5238, p = 0.0072
Spearman correlation: rho = 0.4608, p = 0.0204
Rainfall trend: 1.053 mm/year
NDVI trend: -0.000570 NDVI units/year


Next step — Cell 3: Leave-one-year-out sensitivity test

In [5]:
# Test whether any single year strongly influences the rainfall–NDVI correlation
from scipy.stats import pearsonr

results = []

for year_to_remove in rainfall_ndvi["year"]:
    subset = rainfall_ndvi[rainfall_ndvi["year"] != year_to_remove]

    r, p = pearsonr(
        subset["annual_rainfall_mm"],
        subset["annual_mean_ndvi"]
    )

    results.append({
        "year_removed": int(year_to_remove),
        "pearson_r": r,
        "p_value": p
    })

sensitivity_df = pd.DataFrame(results)

full_r, full_p = pearsonr(
    rainfall_ndvi["annual_rainfall_mm"],
    rainfall_ndvi["annual_mean_ndvi"]
)

sensitivity_df["change_in_r"] = sensitivity_df["pearson_r"] - full_r

print(f"Full-data Pearson r: {full_r:.4f}, p = {full_p:.4f}")
print(f"Lowest leave-one-out r: {sensitivity_df['pearson_r'].min():.4f}")
print(f"Highest leave-one-out r: {sensitivity_df['pearson_r'].max():.4f}")
print(
    "Years removed that make p >= 0.05:",
    sensitivity_df.loc[sensitivity_df["p_value"] >= 0.05, "year_removed"].tolist()
)

print("\nYears with the greatest influence on correlation:")
display(
    sensitivity_df.reindex(
        sensitivity_df["change_in_r"].abs()
        .sort_values(ascending=False).index
    ).head(5).round(4)
)

Full-data Pearson r: 0.5003, p = 0.0109
Lowest leave-one-out r: 0.4219
Highest leave-one-out r: 0.5574
Years removed that make p >= 0.05: []

Years with the greatest influence on correlation:


,year_removed,pearson_r,p_value,change_in_r
15,2015,0.4219,0.0400,-0.0783
13,2013,0.5574,0.0047,0.0571
1,2001,0.4468,0.0286,-0.0534
14,2014,0.5534,0.0050,0.0532
4,2004,0.5530,0.0051,0.0527


Next step — Cell 4: Save the robustness results

In [6]:
# Save leave-one-year-out sensitivity results
sensitivity_df = sensitivity_df.sort_values("year_removed")

output_file = "ghana_rainfall_ndvi_leave_one_year_out_2000_2024.csv"
sensitivity_df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(sensitivity_df))
print("Years tested:", sensitivity_df["year_removed"].min(),
      "to", sensitivity_df["year_removed"].max())

Saved: ghana_rainfall_ndvi_leave_one_year_out_2000_2024.csv
Rows: 25
Years tested: 2000 to 2024


## Research Interpretation and Limitations

### 1. Main Findings

The analysis of annual rainfall and vegetation greenness in Ghana from 2000 to 2024 identified a moderate positive association between rainfall and annual mean NDVI. The original Pearson correlation was \(r = 0.5003\), with \(p = 0.0109\). Spearman's rank correlation was also positive and statistically significant (\(\rho = 0.4577\), \(p = 0.0214\)).

After removing linear time trends from both variables, the positive association remained statistically significant (Pearson \(r = 0.5238\), \(p = 0.0072\); Spearman \(\rho = 0.4608\), \(p = 0.0204\)). This indicates that the observed association was not explained solely by the linear trends in rainfall and NDVI.

The leave-one-year-out sensitivity analysis provided additional evidence of robustness. Across 25 tests, each excluding one year, Pearson's correlation ranged from 0.4219 to 0.5574, and all tests remained statistically significant at the 5% level.

### 2. Interpretation for Agricultural Research

These findings support the use of rainfall variability and satellite-derived vegetation greenness as complementary indicators for investigating vegetation dynamics in Ghana. The positive association is relevant to research on potential forage conditions because water availability can influence vegetation growth.

However, NDVI measures vegetation greenness rather than forage biomass, nutritional quality, or livestock-accessible forage. The results therefore indicate a relationship between rainfall and vegetation greenness, not a direct measurement of forage production or availability.

### 3. Limitations

- **Observational design:** Correlation does not establish causation. Other environmental and land-use factors may influence vegetation greenness.
- **National aggregation:** National annual averages can conceal differences among ecological zones, regions, and local environments.
- **NDVI coverage verification:** The previously calculated valid spatial fraction was 1.0 for every year, but the coverage calculation has not been independently verified. This value should not be interpreted as confirmed complete valid-pixel coverage.
- **Temporal dependence:** The correlation tests do not explicitly account for possible autocorrelation in annual observations.
- **Forage measurement:** NDVI is a proxy for vegetation greenness and does not directly measure forage biomass, quality, or availability to livestock.
- **Scope of inference:** The findings apply to the aggregated annual observations analyzed for 2000–2024 and should not automatically be generalized to individual farms or livestock production systems.

### 4. Conclusion

From 2000 to 2024, annual rainfall and annual mean NDVI in Ghana exhibited a moderate positive association. The relationship remained positive and statistically significant after detrending and in leave-one-year-out sensitivity tests. These robustness checks strengthen confidence that the observed association is not driven solely by linear trends or a single influential year. Further spatially explicit analysis and direct forage measurements would be needed to determine how well these indicators represent actual forage conditions for livestock production.


In [7]:
# Save the final integrated analysis outputs
import pandas as pd

# Save the merged annual dataset
rainfall_ndvi[
    ["year", "annual_rainfall_mm", "annual_mean_ndvi"]
].to_csv(
    "ghana_rainfall_ndvi_merged_2000_2024.csv",
    index=False
)

# Save detrended correlation statistics
detrended_results = pd.DataFrame([
    {
        "method": "Pearson",
        "correlation": pearson_r,
        "p_value": pearson_p
    },
    {
        "method": "Spearman",
        "correlation": spearman_rho,
        "p_value": spearman_p
    }
])

detrended_results.to_csv(
    "ghana_rainfall_ndvi_detrended_correlation_2000_2024.csv",
    index=False
)

# Save the sensitivity results
sensitivity_df.to_csv(
    "ghana_rainfall_ndvi_leave_one_year_out_2000_2024.csv",
    index=False
)

# Confirm saved files
import os

final_files = [
    "ghana_rainfall_ndvi_merged_2000_2024.csv",
    "ghana_rainfall_ndvi_detrended_correlation_2000_2024.csv",
    "ghana_rainfall_ndvi_leave_one_year_out_2000_2024.csv"
]

for file in final_files:
    print(f"{file}: {'Saved' if os.path.exists(file) else 'NOT FOUND'}")

ghana_rainfall_ndvi_merged_2000_2024.csv: Saved
ghana_rainfall_ndvi_detrended_correlation_2000_2024.csv: Saved
ghana_rainfall_ndvi_leave_one_year_out_2000_2024.csv: Saved
